<a href="https://colab.research.google.com/github/nhunggnguyen04/NCKH/blob/main/Phobert_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build Aspect Based Sentiment analysis cho bộ dữ liệu ABSA Tiki book review

- Ở đây mình sẽ fine tune mô hình phobert theo dataset ABSA TIKI BOOK Review (https://www.kaggle.com/datasets/nguynvntnpht/tiki-cleaned-book-reviews/data)
- Sau đó mình sẽ gán nhãn cho các bộ dữ liệu tiki review mình đã crawl từ trước


- Về bộ dữ liệu ABSA mình lấy ở trên Kaggle nó sẽ chia thành các aspect khác nhau (cái này tớ có nói với Nhung về ý tưởng ở trên tv rồi ý) --> thì mấy tuần qua mình đi đọc với học qua về cách fine tune mô hình Phobert✌ Vì tớ đánh giá kết quả của mô hình "wonrax phobert" trước cậu đưa cho tớ chưa được tối ưu vì nhiều nhãn bị gán sai

In [ ]:
# Truy cập drive để lấy dữ liệu
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Tải các thư viện cần thiết
## !pip install transformers
## !pip install torchvision
## !pip install Dataset

## Import mấy cái này nếu cần

!pip install underthesea

In [ ]:
# Import các thư viện cần thiết
import os
os.environ["WANDB_DISABLED"] = "true"
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils import resample
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from datasets import Dataset
from underthesea import word_tokenize
from datasets import Value
print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# Load model sử dụng
MODEL_NAME = "vinai/phobert-base-v2"
TEXT_COL = "content"
# nhãn này sẽ thay đổi trong quá trình mình tune từng model 1 cho từng aspect
LABEL_COL = "as_service"   # đổi lần lượt thành các aspect khác ở trong file data nhen

In [ ]:
# Gán trước mấy cái model để tý xuống dưỡi mình chạy
# Khởi chạy model Phobert
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3    # Có 3 nhãn tương ứng Tích cực, Tiêu cực, Trung tính
)

# Khởi chạy tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Load bộ dữ liệu training
df = pd.read_csv('/content/drive/MyDrive/train_data_clean_dup.csv')

# Fill các dữ liệu thiếu
if 'content' in df.columns:
    df['content'] = df['content'].fillna("")

display(df.head())

,review_id,rating,review_title,product_name,category,content,sentiment_llm,as_content,as_physical,as_price,as_packaging,as_delivery,as_service
0,14357443,4,Hài lòng,Sách Xứ Cát,Sách văn học,"Sản phẩm thật sự chất lượng, và Tiki giao hàng...",1,NaN,NaN,1.0,NaN,2.0,NaN
1,16749123,1,Rất không hài lòng,Bộ sách Làm Giàu Từ Chứng Khoán (How To Make M...,Sách kinh tế,không phù hợp đầu tư dài hạn,0,NaN,NaN,NaN,NaN,NaN,NaN
2,897166,1,Không nên mua,Sách Predictably Irrational : The Hidden Force...,Reference,"Cuốn sách không đúng khổ ghi trên thông tin, k...",0,NaN,2.0,NaN,NaN,NaN,NaN
3,9105241,3,Bình thường,Siêu Cò – How To Be A Power Connector,Sách kinh tế,"Sách hay, nội dung hấp dẫn, thiết thực. Chấm 5...",1,2.0,NaN,NaN,NaN,2.0,NaN
4,6820537,4,Hài lòng,Sách Elon Musk: How The Billionaire CEO Of Spa...,Biographies & Memoirs,sách chất lượng in chưa được cao,1,NaN,1.0,NaN,NaN,NaN,NaN


> Trong dataset thì các aspect sẽ có 3 nhãn tích cực, tiêu cực, trung tính và NaN --> với nhãn Na thì có nghĩa là trong bình luận không đề cập đến aspect ấy

> Mình test các kiểu để cân bằng dữ liệu rồi, đã chuyển 1 nhãn riêng đã chuyển về trung tính nhưng kết quả thường bị overfit

In [ ]:
# Hiển thị bộ dữ liệu chưa lọc NaN
print("Số lượng giá trị NaN trước khi lọc:")
display(df[LABEL_COL].isna().sum())

# Drop dòng có NaN
df = df.dropna(subset=[LABEL_COL])

print("Số lượng giá trị NaN sau khi lọc:")
display(df[LABEL_COL].isna().sum())
print("Phân bố nhãn:")
display(df[LABEL_COL].value_counts())

Số lượng giá trị NaN trước khi lọc:


np.int64(20744)

Số lượng giá trị NaN sau khi lọc:


np.int64(0)

Phân bố nhãn:


,count
as_service,
2.0,338
1.0,176
0.0,134


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import pandas as pd

# Chia tập train/test/validation theo tỷ lệ 80/10/10
# Chia tập gốc (df) thành train (80%) và temp (20%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df[LABEL_COL],
    random_state=42
)

# Chia tiếp temp thành val (10%) và test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df[LABEL_COL],
    random_state=42
)

# Thực hiện Oversampling trên tập train_df (Oversampling giúp cân bằng dữ liệu)
print(f"Oversampling cho tập train, cột: {LABEL_COL}")
max_count = train_df[LABEL_COL].value_counts().max()

resampled_dfs = []
for label in train_df[LABEL_COL].unique():
    df_label = train_df[train_df[LABEL_COL] == label]
    df_label_upsampled = resample(
        df_label,
        replace=True,
        n_samples=max_count,
        random_state=42
    )
    resampled_dfs.append(df_label_upsampled)

# Gộp lại và tráo dữ liệu
train_df = pd.concat(resampled_dfs).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Kích thước tập train (sau oversampling): {len(train_df)}")
print(f"Kích thước tập validation: {len(val_df)}")
print(f"Kích thước tập test: {len(test_df)}\n")

print("Phân bố nhãn trong tập Train:")
display(train_df[LABEL_COL].value_counts())

Oversampling cho tập train, cột: as_service
Kích thước tập train (sau oversampling): 810
Kích thước tập validation: 65
Kích thước tập test: 65

Phân bố nhãn trong tập Train:


,count
as_service,
2.0,270
0.0,270
1.0,270


In [ ]:
# Tách từ có gạch chéo
def vietnamese_word_segmenter(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    try:
        return word_tokenize(text, format='text')
    except Exception as e:
        return str(text)

# Áp dụng tách từ sử dụng segmenter cho Tiếng Việt
train_df['segmented_content'] = train_df['content'].apply(vietnamese_word_segmenter)
val_df['segmented_content'] = val_df['content'].apply(vietnamese_word_segmenter)
test_df['segmented_content'] = test_df['content'].apply(vietnamese_word_segmenter)

# Khởi tạo bộ dữ liệu Hugging Face
train_ds = Dataset.from_pandas(train_df[['content', LABEL_COL]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[['content', LABEL_COL]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[['content', LABEL_COL]], preserve_index=False)

print(train_df[['content', 'segmented_content']].head(1))

                                  content  \
0  Sách ổn! Shipper dễ thương thân thiện!   

                          segmented_content  
0  Sách ổn !_Shipper dễ_thương thân_thiện !  


In [ ]:
# Biến các từ đã được segmented thành các token
def tokenize_function(examples):
    return tokenizer(
        examples['segmented_content'],
        padding='max_length',
        truncation=True,
        max_length=256
    )


# Keep only the necessary columns
train_subset = train_df[['segmented_content', LABEL_COL]]
val_subset = val_df[['segmented_content', LABEL_COL]]
test_subset = test_df[['segmented_content', LABEL_COL]]

# Convert to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_subset, preserve_index=False)
val_dataset = Dataset.from_pandas(val_subset, preserve_index=False)
test_dataset = Dataset.from_pandas(test_subset, preserve_index=False)

# Verify the train dataset
print("Train Dataset:")
print(train_dataset)
print(train_subset.head(1))




Train Dataset:
Dataset({
    features: ['segmented_content', 'as_service'],
    num_rows: 810
})
                          segmented_content  as_service
0  Sách ổn !_Shipper dễ_thương thân_thiện !         2.0


In [ ]:
# Biến bộ dữ liệu thành token
def tokenize_function(examples):
    return tokenizer(
        examples['segmented_content'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Xoá cột text segmented đi để mô hình không nhầm lẫn
tokenized_train = tokenized_train.remove_columns(['segmented_content'])
tokenized_val = tokenized_val.remove_columns(['segmented_content'])
tokenized_test = tokenized_test.remove_columns(['segmented_content'])

# Verify
print("Tokenized Train Dataset Features:", tokenized_train.features)
print("\nFirst row of tokenized train dataset:")
print(tokenized_train[0])

Map:   0%|          | 0/810 [00:00<?, ? examples/s]

Map:   0%|          | 0/65 [00:00<?, ? examples/s]

Map:   0%|          | 0/65 [00:00<?, ? examples/s]

Tokenized Train Dataset Features: {'as_service': Value('float64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}

First row of tokenized train dataset:
{'as_service': 2.0, 'input_ids': [0, 6042, 4752, 3, 2333, 7220, 18467, 4924, 3188, 381, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

> Oke đến bước này là có thể fine-tune mô hình từ bộ dữ liệu mình có được rồi

In [ ]:
# các parameter để train
training_args = TrainingArguments(
    output_dir="./sentiment_class",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    logging_dir="./train_logs",
    logging_steps=100,
    fp16=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# F1_score, Accuracy
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [ ]:
import torch
from torch import nn
from transformers import Trainer, EarlyStoppingCallback
from datasets import Value
import numpy as np
from sklearn.utils import resample

# Rename and cast labels
if LABEL_COL in tokenized_train.column_names:
    tokenized_train = tokenized_train.rename_column(LABEL_COL, "labels")
if LABEL_COL in tokenized_val.column_names:
    tokenized_val = tokenized_val.rename_column(LABEL_COL, "labels")

tokenized_train = tokenized_train.cast_column("labels", Value("int64"))
tokenized_val = tokenized_val.cast_column("labels", Value("int64"))

# 3. Standard Trainer (No custom weighted loss)
model.config.problem_type = "single_label_classification"

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\nRestarting training with oversampled data and standard loss...")
trainer.train()

Casting the dataset:   0%|          | 0/810 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/65 [00:00<?, ? examples/s]


Restarting training with oversampled data and standard loss...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.718408,0.830769,0.790171
2,No log,0.417373,0.861538,0.823721
3,No log,0.256248,0.953846,0.939469
4,1.044072,0.189425,0.953846,0.939469
5,1.044072,0.161621,0.969231,0.964161
6,1.044072,0.167098,0.969231,0.961692
7,1.044072,0.166334,0.969231,0.961692
8,0.092103,0.167709,0.969231,0.961692


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=208, training_loss=0.5477045742938151, metrics={'train_runtime': 70.2757, 'train_samples_per_second': 115.26, 'train_steps_per_second': 3.7, 'total_flos': 852487473438720.0, 'train_loss': 0.5477045742938151, 'epoch': 8.0})

In [ ]:
# Ensure the label column is named 'labels' for the trainer to recognize it as ground truth
if LABEL_COL in tokenized_test.column_names:
    tokenized_test_final = tokenized_test.rename_column(LABEL_COL, "labels")
else:
    tokenized_test_final = tokenized_test

# Cast the labels to int64 to avoid NotImplementedError for Float
tokenized_test_final = tokenized_test_final.cast_column("labels", Value("int64"))

# Predict using the renamed dataset
test_results = trainer.predict(tokenized_test_final)
pred_labels = np.argmax(test_results.predictions, axis=1)
true_labels = test_results.label_ids

# Check if true_labels exist before calculating metrics (using None in Python)
if true_labels is not None:
    test_acc = accuracy_score(true_labels, pred_labels)
    print(f"Độ chính xác (accuracy) trên tập test {LABEL_COL}: {test_acc:.4f}")

    class_names = ["Tiêu cực", "Trung lập", "Tích cực"]
    print(f"Báo cáo phân loại trên tập test {LABEL_COL}:")
    print(classification_report(true_labels, pred_labels, target_names=class_names))
else:
    print("Lỗi: Không tìm thấy nhãn thực tế (true labels) trong dataset.")

Casting the dataset:   0%|          | 0/65 [00:00<?, ? examples/s]

Độ chính xác (accuracy) trên tập test as_service: 0.9231
Báo cáo phân loại trên tập test as_service:
              precision    recall  f1-score   support

    Tiêu cực       0.82      1.00      0.90        14
   Trung lập       0.88      0.88      0.88        17
    Tích cực       1.00      0.91      0.95        34

    accuracy                           0.92        65
   macro avg       0.90      0.93      0.91        65
weighted avg       0.93      0.92      0.92        65



In [ ]:
# Lưu lại mô hình đã được fine-tune (tốt nhất) và tokenizer
trainer.save_model("./best_phobert_model")
tokenizer.save_pretrained("./best_phobert_model")

# In ra kết quả mô hình tốt nhất trên tập validation
best_acc = trainer.state.best_metric
print(f"Độ chính xác tốt nhất trên tập validation: {best_acc:.4f}")
print("Checkpoint mô hình tốt nhất được lưu tại:", trainer.state.best_model_checkpoint)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Độ chính xác tốt nhất trên tập validation: 0.9692
Checkpoint mô hình tốt nhất được lưu tại: ./sentiment_class/checkpoint-130


#Gán nhãn bộ dữ liệu tiki gốc

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from underthesea import word_tokenize
from tqdm.auto import tqdm

# 1. Tải mô hình và tokenizer
model_dir = "./best_phobert_model"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 2. Đọc dữ liệu
df_unlabel = pd.read_csv('/content/drive/MyDrive/tiki_reviews_with_sentiment.csv')
# Xử lý giá trị rỗng
df_unlabel['content'] = df_unlabel['content'].fillna('')

# 3. Hàm tiền xử lý (tách từ)
def vietnamese_word_segmenter(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    try:
        return word_tokenize(text, format='text')
    except Exception:
        return str(text)

# Vì chạy lần trước đã segmented từ rồi nên không cần nữa
#print("Đang tách từ...")
#df_unlabel['segmented_content'] = df_unlabel['content'].apply(vietnamese_word_segmenter)

# 4. Tạo Pytorch Dataset để xử lý batch
class ReviewDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0)
        }

dataset = ReviewDataset(df_unlabel['segmented_content'].tolist(), tokenizer)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

# 5. Dự đoán
all_preds = []
all_scores = []

print("Đang dự đoán cảm xúc...")
with torch.no_grad():
    for batch in tqdm(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)

        # Lấy nhãn có xác suất cao nhất
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        # Tính sentiment score (ví dụ: xác suất của lớp được chọn)
        scores = torch.max(probs, dim=1).values.cpu().numpy()

        all_preds.extend(preds)
        all_scores.extend(scores)

# 6. Gán nhãn và lưu kết quả

label_id = LABEL_COL + "_label_id"
sentiment_score = LABEL_COL + "_sentiment_score"
df_unlabel[label_id] = all_preds
df_unlabel[sentiment_score] = all_scores

# Lưu lại file
save_path = '/content/drive/MyDrive/tiki_reviews_with_sentiment.csv'
df_unlabel.to_csv(save_path, index=False)
print(f"Đã lưu kết quả tại: {save_path}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Đang dự đoán cảm xúc...


  0%|          | 0/2406 [00:00<?, ?it/s]

Đã lưu kết quả tại: /content/drive/MyDrive/tiki_reviews_with_sentiment.csv


In [ ]:
df_unlabel[label_id].value_counts()

,count
as_service_label_id,
2,60596
1,8419
0,7963
